[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/outputs/Document_Outputs_Detect.ipynb)


## Jump to an API


### outputs.Box


### outputs.Detection


### outputs.Detections


### outputs.build_detections


`Box` stores one annotation, `Detection` stores one frame, and `Detections` stores an ordered prediction batch.


### Combine predictions safely

`Detection.combine(other, method="union", iou=0.5)` mutates and returns the left frame.
Both frames must have the same identity and image_size, including for append mode.
Pairs are selected greedily by descending IoU and must share a label. Unmatched
boxes are retained. Methods: union, intersection, average, weighted, first,
second, append. Matched scores use max; new UIDs and combination metadata replace
previous matched metadata and track IDs. Intersection only changes matched geometry.

`Detections.combine` aligns frames by identity, validates all shared image sizes
before changing boxes, and clones frames unique to the other collection. Both
collections must be materialized. Regular and packed storage are supported.


In [ ]:
from klygo import outputs
left = outputs.Detection(
    [outputs.Box((0, 0, 20, 20), label="cat", score=0.9)],
    source_id="combine-example", image_size=(100, 100),
)
right = outputs.Detection(
    [outputs.Box((2, 2, 22, 22), label="cat", score=0.8)],
    source_id="combine-example", image_size=(100, 100),
)
left.combine(right, method="weighted", iou=0.5)
print(left.boxes)
first = outputs.Detections([left], packed=True)
second = outputs.Detections([right])
first.combine(second, method="union")
print(first[0].boxes)
try:
    left.combine(outputs.Detection([], source_id="different", image_size=(100, 100)))
except ValueError as error:
    print(error)


### Source images and direct comparison

Detection.image returns an independent, unannotated PIL image from the in-memory
source or source_path. Video source paths use frame_index and release the reader
after copying pixels. Missing sources raise RuntimeError; missing files raise
FileNotFoundError. Compare two frames directly with visual.compare_images; use
mode="show" or mode="save" with output_path.


In [ ]:
from PIL import Image
from klygo import outputs, visual
first = outputs.Detection([], source=Image.new("RGB", (80, 60), "red"), image_size=(80, 60))
second = outputs.Detection([], source=Image.new("RGB", (100, 70), "blue"), image_size=(100, 70))
print(first.image.size)
preview = visual.compare_images(first, second, mode="show")
print(preview.size)


### Save uniform annotated images and videos

Both save_images and save_video return None. Pass size=(width, height) to
letterbox the rendered image, so annotation outlines and labels scale with it.
The source image and box coordinates remain unchanged. Without size, images
retain their sizes and video frames must have equal dimensions.


In [ ]:
import tempfile
from PIL import Image
from klygo import files, media, outputs
with tempfile.TemporaryDirectory() as temporary:
    export_root = files.path(temporary)
    annotated = outputs.Detections([
        outputs.Detection([outputs.Box((5, 5, 15, 15), label="cat")],
            source=Image.new("RGB", size, "white"), source_id=str(i), image_size=size)
        for i, size in enumerate(((80, 40), (40, 80)))
    ])
    annotated.save_images(export_root / "images", size=(64, 64), extension="png")
    annotated.save_video(export_root / "video.mp4", size=(64, 64), fps=5)
    print(media.info(export_root / "video.mp4")["size"])


### Device-preserving arrays

Detection.to_arrays returns independent boxes, scores, class_ids and labels.
NumPy/CuPy arrays retain storage device and precision. No pixels are decoded
or CUDA arrays transferred to CPU. Metrics consumes these arrays directly.


In [ ]:
from klygo import outputs
array_frame = outputs.Detection([outputs.Box((0, 0, 10, 10), label="cat")], image_size=(20, 20))
arrays = array_frame.to_arrays()
arrays["boxes"][0, 0] = 5
print(array_frame.boxes, arrays["boxes"])


## Put it together


Use `model.predict(...)`, mutate the returned results, then render or export them.


## Remember


Mutating APIs return `None`; frame and box identities remain stable.
